In [ ]:
# Copyright 2026 Arm Limited and/or its affiliates.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

# Ethos-U55 with CMSIS-NN fallback example

This guide demonstrates the current full flow for handling operators which does not lower
to the Ethos-U55 using the Cortex-M backend to make sure they use accelerated CMSIS-NN implementations. 
The basic idea is that the Ethos-U backend will reject any nodes which are not supported,
leaving them to be handled by the Cortex-M backend.

Before you begin: Make sure you have completed the `ethos_u_minimal_example` for a
basic understanding of the Ethos-U backend and have your environment setup. 


*Some scripts in this notebook produces long output logs: Configuring the 'Customizing Notebook Layout' settings to enable 'Output:scrolling' and setting 'Output:Text Line Limit' makes this more manageable*

## Setup

The first step is creating a simple model which does not fully lower to the Ethos-U55.
Importantly it is exported with channels_last data, since the Cortex-M backend currently
only supports lowering operators in that data-format.  

Constraints for the basic operations performed by the Ethos-U55 can be found in the
[Ethos-U Vela repository](https://gitlab.arm.com/artificial-intelligence/ethos-u/ethos-u-vela/-/blob/main/SUPPORTED_OPS.md?ref_type=heads#ethos-u55-and-ethos-u65-tosa-conv2d-constraints). Note that the listed operators does not map exactly to PyTorch operators, but rather a subset found in
the graph after decompositions in the Ethos-U backend.

In [ ]:
import torch
from executorch.backends.arm.ethosu import EthosUCompileSpec, EthosUPartitioner
from executorch.backends.arm.quantizer import (
    EthosUQuantizer,
    get_symmetric_quantization_config,
)
from executorch.backends.cortex_m.passes.cortex_m_pass_manager import CortexMPassManager
from executorch.exir import (
    EdgeCompileConfig,
    ExecutorchBackendConfig,
    to_edge_transform_and_lower,
)
from executorch.extension.export_util.utils import save_pte_program
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e

target = "ethos-u55-128"
output_path = "ethos_u_cmsis_nn_fallback_example.pte"

class ToyMixedModule(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(
            in_channels=3,
            out_channels=4,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False,
        )
        self.conv2 = torch.nn.Conv2d(
            in_channels=4,
            out_channels=1,
            kernel_size=3,
            stride=4,
            padding=1,
            bias=False,
        ) # Stride=4 not supported on Ethos-U55

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv1(x)
        x = torch.relu(x)
        return self.conv2(x)

model = ToyMixedModule().eval().to(memory_format=torch.channels_last)
example_inputs = (
    torch.randn(1, 3, 8, 8, dtype=torch.float32).to(memory_format=torch.channels_last),
)
exported_program = torch.export.export(model, example_inputs)
exported_program.module().graph.print_tabular()

# Ethos-U lowering

The Ethos-U lowering of the model is identical to the minimal example, and as expected
the printed graph leaves the regular `torch.nn.Conv2d` with `stride=4` and some quantization/dequantization nodes
outside of the Ethos_u call_delegate operator. 

One important part in this step is that this `torch.nn.Conv2d` with `stride=4` has been quantized to
a format supported by the Cortex-M backend by the Ethos-U quantizer even if it was not
delegated, since the Cortex-M backend will only lower correctly quantized operators. Would there be
a discrepancy, see the [quantizer tutorial](https://github.com/pytorch/executorch/blob/main/examples/arm/quantizer_tutorial.ipynb) for
how to configure more precise quantization.

In [ ]:
compile_spec = EthosUCompileSpec(target=target)
quantizer = EthosUQuantizer(compile_spec)
quantizer.set_global(get_symmetric_quantization_config(is_per_channel=True))

prepared = prepare_pt2e(exported_program.module(), quantizer)
prepared(*example_inputs)
quantized_model = convert_pt2e(prepared)
quantized_exported_program = torch.export.export(quantized_model, example_inputs)

edge_program_manager = to_edge_transform_and_lower(
    quantized_exported_program,
    partitioner=[EthosUPartitioner(compile_spec)],
    compile_config=EdgeCompileConfig(_check_ir_validity=False),
)

edge_program_manager.exported_program().graph_module.graph.print_tabular()

# Cortex-M lowering

Finally the Cortex-M backend is applied, and the graph is now fully accelerated. The
`cortex_m_kernels` can be spotted in the printed graph.

In [ ]:

edge_program_manager._edge_programs["forward"] = CortexMPassManager(
     edge_program_manager.exported_program()
).transform()

executorch_program = edge_program_manager.to_executorch(
     config=ExecutorchBackendConfig(extract_delegate_segments=False)
)
save_pte_program(executorch_program, output_path)

edge_program_manager.exported_program().graph_module.graph.print_tabular()


## Build

The executor runner is built as usual, making sure to link the Cortex-M dependencies. In the available
example executor_runner CMakeFile this is already done, with the Cortex-M kernel and kernel registration libraries
`cortex_m_kernels` and `cortex_m_ops_lib` corresponding to `portable_kernels` and `arm_portable_ops_lib` for the the
unaccelerated portable kernels. For more information about kernel registration, see the
[documentation](https://docs.pytorch.org/executorch/stable/kernel-library-custom-aten-kernel.html).


In [ ]:
%%bash 
source arm-scratch/setup_path.sh
# Ensure CMake resolves the ExecuTorch checkout root regardless of caller env
export EXECUTORCH_ROOT=$(cd ../.. && pwd)

# Build example executor runner application to examples/arm/ethos_u_cmsis_nn_fallback_example
cmake -DCMAKE_TOOLCHAIN_FILE=$(pwd)/ethos-u-setup/arm-none-eabi-gcc.cmake \
      -DCMAKE_BUILD_TYPE=Release \
      -DET_PTE_FILE_PATH=ethos_u_cmsis_nn_fallback_example.pte \
      -DTARGET_CPU=cortex-m55 \
      -DETHOSU_TARGET_NPU_CONFIG=ethos-u55-128 \
      -DMEMORY_MODE=Shared_Sram \
      -DSYSTEM_CONFIG=Ethos_U55_High_End_Embedded \
      -Bethos_u_cmsis_nn_fallback_example \
      -S executor_runner/standalone
cmake --build ethos_u_cmsis_nn_fallback_example -j$(nproc) -- arm_executor_runner

## Sanity check output

In [ ]:
import subprocess
import re

# Use quantized model in eager mode as reference. By default the executor runner will use 1:s as input.
test_inputs = (torch.ones_like(example_inputs[0]),)
reference_result = quantized_exported_program.module()(*test_inputs).flatten().tolist()

# Run the lowered .pte file on FVP using helper script and extract the output numbers using regex
fvp_output = subprocess.run("../../backends/arm/scripts/run_fvp.sh --elf=ethos_u_cmsis_nn_fallback_example/arm_executor_runner --target=ethos-u55-128", shell=True, capture_output=True)
lowered_result = [float(x) for x in re.findall("-?\d\.\d{6}" , str(fvp_output.stdout))]

print(reference_result)
print(lowered_result)
